<a href="https://colab.research.google.com/github/Moulicodes/movie-recommendation-system/blob/main/movie_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_movies = pd.read_csv('/content/tmdb_5000_movies.csv')

In [ ]:
df_movies.head(1)

In [ ]:
df_movies.head(1)['genres'].values

In [ ]:
df_movies.head(1)['keywords'].values

In [ ]:
df_movies.head(1)['production_companies'].values

In [ ]:
df_movies.head(1)['production_countries'].values

In [ ]:
df_movies['spoken_languages'].value_counts()

In [ ]:
df_movies['status'].value_counts()

In [ ]:
df_movies[df_movies['original_title'] != df_movies['title']].shape

In [ ]:
df_movies[['vote_average', 'vote_count', 'popularity', 'title']].sample(5)

In [ ]:
sns.histplot(df_movies['original_language'])
plt.xticks(rotation = 'vertical')
plt.show()

In [ ]:
import ast

df_movies['keywords'] = df_movies['keywords'].apply(ast.literal_eval)
df_movies['genres'] = df_movies['genres'].apply(ast.literal_eval)
df_movies['production_companies'] = df_movies['production_companies'].apply(ast.literal_eval)


In [ ]:
def transform_listofdicts(listofdicts):
  str = ""
  for dict in listofdicts:
    str += dict['name'].lower().replace(" ", "") + " "
  return str.strip()

In [ ]:
df_movies['genres'] = df_movies['genres'].apply(transform_listofdicts)
df_movies['keywords'] = df_movies['keywords'].apply(transform_listofdicts)
df_movies['production_companies'] = df_movies['production_companies'].apply(transform_listofdicts)

In [ ]:
df_movies = df_movies.drop(columns = ['budget', 'homepage', 'original_language', 'original_title', 'production_countries', 'release_date', 'revenue', 'spoken_languages', 'status', 'vote_count' ])
df_movies.shape

In [ ]:
df_movies.head(1)

In [ ]:
df_credits = pd.read_csv('/content/tmdb_5000_credits.csv')

In [ ]:
df_credits.head()

In [ ]:
df_credits['cast'] = df_credits['cast'].apply(ast.literal_eval).apply(transform_listofdicts)
df_credits['crew'] = df_credits['crew'].apply(ast.literal_eval).apply(transform_listofdicts)

In [ ]:
df_credits.shape

In [ ]:
df_movies.head()

In [ ]:
df_credits.head()

In [ ]:
df = df_movies.merge(df_credits, left_on = 'id', right_on = 'movie_id')
df = df.drop(columns = ['movie_id', 'title_y'])
df = df.rename(columns = {'title_x': 'title'})
df.head(1)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df[["overview", "tagline"]] = df[["overview", "tagline"]].fillna("")
df["runtime"] = df["runtime"].fillna(0)

In [ ]:
df.info()

In [ ]:
df.head(3)

In [ ]:
!pip install nltk

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


In [ ]:
stopwords.words('english')

In [ ]:
import string
string.punctuation

In [ ]:
def preprocess_text(text):
  text = text.lower()
  words = nltk.word_tokenize(text)
  processed_text = ""
  for word in words:
    if (word not in stopwords.words('english')) and (not word.isnumeric()) and (word not in string.punctuation):
      processed_text += word + " "

  return processed_text.strip()

In [ ]:
# preprocess_text('In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.')

df['overiew'] = df['overview'].apply(preprocess_text)
df['overiew'].head()

In [ ]:
df['tagline'] = df['tagline'].apply(preprocess_text)
df['tagline'].head()

In [ ]:
df['processed_description'] = df['genres'] + " " + df['keywords'] + " " + df['overview'] + " " + df['tagline'] + " " + df['cast'] + df['crew']
df['processed_description'].head(1).values

1. TF-IDF Vectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words = 'english')
tfidf_matrix = tfidf.fit_transform(df['processed_description'])
tfidf_matrix.shape

Because we are using tfidf vectorizer there's no need to explicitly remove stop words, punctuations or convert texts to lower case as it will handle all these functions by itself. But we still need to explicitly implement space collapsing wherever necessary. For example, removing the space between first and last name.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

tfidf_cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
tfidf_cosine_sim_rounded = tfidf_cosine_sim.round(3)
tfidf_cosine_sim_rounded.shape

In [ ]:
df.iloc[[2]][['title', 'genres', 'keywords', 'overview', 'tagline', 'production_companies', 'cast', 'crew']]

In [ ]:
np.where((tfidf_cosine_sim_rounded[2] > 0.1) & (tfidf_cosine_sim_rounded[2] < 0.99))[0]

In [ ]:
tfidf_cosine_sim_rounded[2][(tfidf_cosine_sim_rounded[2] > 0.1) & (tfidf_cosine_sim_rounded[2] < 0.99)]

In [ ]:
df.iloc[[11, 29]][['title', 'genres', 'keywords', 'overview', 'tagline', 'production_companies', 'cast', 'crew']]

**Problem with tfidf vectorizer:**

It penalizes the words present in genres, keywords, cast and crew if they occur very frequently in multiple movies. But the more matching the genres, keywords, cast and crew of various movies, the greater should be the similarity scores between them. But because tfidf penalizes the frequently occuring words, it behaves counter-productive to our goal.**bold text**

2. Count Vectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(stop_words = 'english')
count_matrix = count_vectorizer.fit_transform(df['processed_description'])
count_matrix.shape

In [ ]:
count_cosine_sim = cosine_similarity(count_matrix, count_matrix)
count_cosine_sim_rounded = count_cosine_sim.round(3)
count_cosine_sim_rounded.shape

In [ ]:
df.iloc[[2]][['title', 'genres', 'keywords', 'overview', 'tagline', 'production_companies', 'cast', 'crew']]

In [ ]:
np.where((count_cosine_sim_rounded[2] > 0.1) & (count_cosine_sim_rounded[2] < 0.99))[0]

In [ ]:
count_cosine_sim_rounded[2][(count_cosine_sim_rounded[2] > 0.1) & (count_cosine_sim_rounded[2] < 0.99)]

So, there are four other movies that share a decent similarity score with the movie Spectre.

In [ ]:
df.iloc[np.where((count_cosine_sim_rounded[2] > 0.1) & (count_cosine_sim_rounded[2] < 0.99))[0]][['title', 'genres', 'keywords', 'overview', 'tagline', 'production_companies', 'cast', 'crew']]

**Problem with count vectorizer:**

Consider two completely unrelated action movies.

Movie A (James Bond): "A man goes on a mission to save the world from danger."

Movie B (Avatar): "A man on a foreign planet tries to save his new world from danger."

Both movies heavily feature highly common narrative words: "man", "save", "world", "danger"

So, both of these movies are likely to have high similarity scores even though they are completely unrelated films